# Fine-Grained Damage Severity Classification — Template

Target classes proposed for the capstone:

- Good
- Slightly Worn
- Torn
- Dirty / Soiled
- Crumpled
- Missing Corner
- Heavily Damaged

## Critical labeling requirement

The recommended 5,125-image spoilt-banknote dataset documents damaged notes but its released class labels are denomination/old-new categories, **not these seven damage classes**.

Train this notebook only after the images have been manually or otherwise reliably annotated into the seven classes.


In [ ]:
from pathlib import Path
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = Path("data/damage_severity")
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42


In [ ]:
if not DATA_DIR.exists():
    raise FileNotFoundError(
        "Create data/damage_severity/<damage_class>/... after verified annotation."
    )

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
print("Damage classes:", class_names)


In [ ]:
augmentation = keras.Sequential([
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.10),
    layers.RandomContrast(0.10),
])

base = tf.keras.applications.MobileNetV2(
    include_top=False,
    weights="imagenet",
    input_shape=IMG_SIZE + (3,)
)
base.trainable = False

inputs = keras.Input(shape=IMG_SIZE + (3,))
x = augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(len(class_names), activation="softmax")(x)

model = keras.Model(inputs, outputs)
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=[
        keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True),
        keras.callbacks.ModelCheckpoint(
            "models/damage_severity_model.keras",
            save_best_only=True
        )
    ]
)


## Recommendation layer

The application should clearly separate:

1. **Model output** — predicted damage class and confidence.
2. **Informational guidance** — a general message based on official RBI note-exchange rules.

The model must not claim to determine refund value, legal validity, or final exchange eligibility.
